# Strategy Order Payload Analysis
Investigating how many payloads are sent at once with alerts.py, need to understand valid order threshold to prevent stale alerts from being executed

In [ ]:
# config.py — MYM Signal Alert System
# ---------------------------------------------------------------
# Secrets and environment-specific paths are loaded from the
# environment — NOT hard-coded here. Same file runs unchanged on
# your Mac (.env) and on the cloud host (Variables/Secrets UI).
# Never commit real secrets to this file.
# ---------------------------------------------------------------

import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # cloud host sets env vars directly

def _require(name):
    """Return a required env var, or fail loudly with a clear message."""
    val = os.getenv(name)
    if not val:
        raise RuntimeError(
            f"Missing required environment variable: {name}. "
            f"Set it in your local .env or the cloud host's environment. "
            f"See .env.example for the full list."
        )
    return val


# ── PATHS (environment-overridable) ───────────────────────────
BASE      = Path(os.getenv("MYM_BASE", os.path.expanduser("~/prod/em_algo")))
DATA_DIR  = BASE / "data"
DATA_1M   = str(DATA_DIR / "MYM_1m_tv_bento.csv")   # backtest base bars
LOG_FILE  = str(BASE / "logs" / "signal_log.txt")

# ── STRATEGY PARAMETERS (v6 — must match strategy.py backtest) ─
TICKER            = "MYM"
ACCOUNT_SIZE      = 50000
RISK_PCT          = 0.005      # 0.5% per trade (v7 backtest value)
TICK_VALUE        = 0.50       # MYM = $0.50/point (CME-verified; YM is $5)
MIN_SCORE         = 7
T2_CAP            = 2.5          # R:R cap on T2
MAX_BODY_PCT      = 30         # base candle: body <= 40% of range
MIN_BODY_PCT      = 35         # leg candle:  body >= 50% of range
MIN_RANGE_RATIO   = 0.631      # leg range >= base_height * this
RANGE_MULTIPLIER  = 0.36       # base range <= rolling-median * this
MIN_ZONE_HEIGHT   = 30         # pts
MAX_BASE_CANDLES  = 6
STOP_BUFFER       = 8          # static fallback; live run computes dynamically

# Lookbacks — note the UNITS, they differ:
ZONE_LOOKBACK_DAYS  = 60       # set_curve: last 60 x 15m bars
TREND_WINDOW        = 120       # classify_trend: rolling window of (last 10hrs) 5m bars
ZONE_WINDOW_DAYS    = 120       # detect_zones: last 120 candles days (last 10hrs) of 5m

# ── SCHEDULER ─────────────────────────────────────────────────
RUN_HOURS = [8, 9, 10, 11, 12]
TIMEZONE  = "US/Eastern"

# ── SECRETS (loaded from environment — never hard-coded) ──────
EMAIL_SENDER   = os.getenv("EMAIL_SENDER", "")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")   # Gmail APP password
EMAIL_RECEIVER = os.getenv("EMAIL_RECEIVER", EMAIL_SENDER)

# Discord: channel settings -> Integrations -> Webhooks -> copy URL
DISCORD_WEBHOOK_URL = os.getenv("DISCORD_WEBHOOK_URL")

# TradersPost: strategy webhook URL (used by alerts.traderspost_alert)
TRADERSPOST_WEBHOOK_URL = os.getenv("TRADERSPOST_WEBHOOK_URL", "")

# ── DATABENTI API CREDS ──────
DATABENTO_API_KEY = os.getenv("DATABENTO_API_KEY", "")

# ── CHARLES SCHWAB TRADER API CREDS ──────
SCHWAB_API_KEY          = "gAvjFUyE1AiLJIccGQ0DrsoYwfqOIXfidEJ3moGUKSHwPHxb" 
SCHWAB_APP_SECRET       = "AgmDCFGiwoaSjs0IZfEuAhdnF6H0Ad9qJn3sLAtsXQKzL02Xy9PGii4D9jgrsTAm" 
SCHWAB_CALLBACK_URL     = "https://127.0.0.1:8182" 
SCHWAB_TOKEN_PATH       = "./schwab_token.json"
SCHWAB_HEADLESS= ""

# Live data providers (optional until activated)
#ALPACA_KEY    = os.getenv("ALPACA_KEY", "")
#ALPACA_SECRET = os.getenv("ALPACA_SECRET", "")

# Secrets required ONLY to send alerts. Enforced at send time, not import.
ALERT_REQUIRED = ["EMAIL_SENDER", "EMAIL_PASSWORD", "DISCORD_WEBHOOK_URL",
                  "TRADERSPOST_WEBHOOK_URL"]

def require_alert_secrets():
    """Raise if any alert secret is missing. Call before sending alerts."""
    missing = [k for k in ALERT_REQUIRED if not globals().get(k)]
    if missing:
        raise RuntimeError(
            f"Cannot send alert — missing secrets: {', '.join(missing)}. "
            f"Set them in your local .env or the cloud host's environment. "
            f"(Backtests/notebooks do not need these.)"
        )


In [ ]:
# verify_env.py — confirm all required secrets are present.
# Prints only PRESENCE, never the values. Run after setting up your
# .env (local) or the host environment (cloud).
#
#   python verify_env.py

import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

REQUIRED = ["EMAIL_SENDER", "EMAIL_PASSWORD", "DISCORD_WEBHOOK_URL"]
OPTIONAL = ["EMAIL_RECEIVER", "MYM_BASE", "ALPACA_KEY", "ALPACA_SECRET",
            "MASSIVE_API_KEY", "DATABENTO_API_KEY"]

print("Required:")
for k in REQUIRED:
    print(f"  {'OK ' if os.getenv(k) else 'MISSING'}  {k}")

print("Optional:")
for k in OPTIONAL:
    print(f"  {'set' if os.getenv(k) else '- - '}  {k}")

missing = [k for k in REQUIRED if not os.getenv(k)]
if missing:
    raise SystemExit(f"\nFAIL — missing required: {', '.join(missing)}")

print("\nAll required secrets present. Safe to run signal_check.py.")


In [ ]:
# alerts.py - Valid Zone & Entry Signal Workflow

from __future__ import annotations

import os
import json
import logging
import smtplib
from dataclasses import dataclass
from typing import Any, Mapping
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import requests
from dotenv import load_dotenv

load_dotenv()
import config

logger = logging.getLogger(__name__)
# ─────────────────────────────────────────────
# alerts.py - MYM Signal Alert System
# Email via Gmail SMTP + Push notifications via Discord Webhook + TradersPost Send
# ─────────────────────────────────────────────

# DECLARE TEST MODE
# test_mode: when True, adds "test": true so TradersPost validates only.
test_mode = True

# ── EMAIL ─────────────────────────────────────
def send_email(subject, body):
    try:
        msg = MIMEMultipart()
        msg["From"]    = config.EMAIL_SENDER
        msg["To"]      = config.EMAIL_RECEIVER
        msg["Subject"] = subject
        msg.attach(MIMEText(body, "plain"))

        with smtplib.SMTP("smtp.gmail.com", 587) as server:
                    server.ehlo()
                    server.starttls()
                    server.login(config.EMAIL_SENDER, config.EMAIL_PASSWORD)
                    server.sendmail(config.EMAIL_SENDER, config.EMAIL_RECEIVER, msg.as_string())

        print("✅ Email sent successfully")
        return True

    except Exception as e:
        print(f"❌ Email failed: {e}")
        return False

# ── DISCORD ──────────────────────────────────

def discord_alert(message, alert_type="signal"):
    """
    Send alert via Discord webhook.
    
    Args:
        message (str): The alert message (entry, stop, contracts, etc.)
        alert_type (str): Type of alert ("signal", "fill", "loss", etc.)
    
    Returns:
        bool: True if sent successfully, False if failed
    """
    webhook_url = os.getenv("DISCORD_WEBHOOK_URL")
    
    if not webhook_url:
        print("❌ DISCORD_WEBHOOK_URL not found in .env")
        return False
    
    try:
        # Format the message with emoji and type
        emoji = {
            "signal": "🎯",
            "fill": "✅",
            "loss": "❌",
            "warning": "⚠️",
            "update": "📊"
        }.get(alert_type, "📢")
        
        # Build Discord message
        discord_message = f"{emoji} **{alert_type.upper()}**\n\n{message}"
        
        # Send via webhook
        payload = {"content": discord_message}
        response = requests.post(webhook_url, json=payload, timeout=5)
        
        # Check if successful (204 = No Content, which is expected)
        if response.status_code == 204:
            print(f"✅ Discord alert sent: {alert_type}")
            return True
        else:
            print(f"⚠️ Discord returned status {response.status_code}")
            return False
    
    except requests.exceptions.Timeout:
        print("❌ Discord request timed out")
        return False
    except Exception as e:
        print(f"❌ Discord send failed: {e}")
        return False


# ── TRADERSPOST PAPER TRADING ──────────────────────────────────
# """TradersPost webhook client for sending OTA trade signals.

# All payload fields and response shapes are confirmed against the official
# TradersPost OpenAPI specification:
# https://docs.traderspost.io/docs/developer-resources/webhook-reference

# Spec-confirmed invariants this module depends on:
#   * Required request fields: "ticker", "action" (action enum: buy/sell/exit/cancel/add).
#   * Field names are case-sensitive.
#   * "signalPrice" is required when the broker has no market data (Tradovate/Lucid).
#   * "test": true validates the payload without submitting an order.
#   * Non-standard data must go under "extras" to avoid colliding with reserved fields.
#   * 200 success body: {success, id, logId, payload}.
#   * 400 error body: {success, messageCode, message}.
# """
# --------------------------------------------------------------------------- #
# Configuration (surfaced as constants so a maintainer changes them in one place)
# --------------------------------------------------------------------------- #
test_mode = test_mode
DEFAULT_TICKER = "MYM"
DEFAULT_TIMEOUT = 10  # seconds

# Signal types that mean "go long". Demand zones (drop-base-rally) buy;
# everything else sells. Extend this set as new long patterns are added.
LONG_SIGNAL_TYPES = frozenset({"DBR"})

# Internal signal keys that have no reserved TradersPost field and should be
# forwarded as opaque metadata under "extras".
EXTRA_KEYS = ("formed_at", "score","untouched?","stop_points", "total_risk", "potential_t1", "potential_t2")
    
# Internal signal keys that MUST be present to build a valid order.
REQUIRED_KEYS = ("type", "entry", "stop", "t1_points", "t2_points", "contracts")

# --------------------------------------------------------------------------- #
# Result type (so callers branch on structured data, never on log strings)
@dataclass(frozen=True)
class AlertResult:
    # """Outcome of a webhook send.

    # Attributes:
    #     ok:           True only on HTTP 200.
    #     status_code:  HTTP status, or None if the request never completed.
    #     signal_id:    TradersPost signal id from a 200 body (for reconciliation).
    #     message_code: Structured error code from a 4xx body (for branching).
    #     detail:       Human-readable message for logs/alerts.
    # """

    ok: bool
    status_code: int | None = None
    signal_id: str | None = None
    message_code: str | None = None
    detail: str = ""

# --------------------------------------------------------------------------- #
# Validation + payload construction (pure, no I/O — unit-testable in isolation)
# --------------------------------------------------------------------------- #

class SignalError(ValueError):
    """Raised when an internal signal cannot form a valid TradersPost order."""


def validate_signal(signal: Mapping[str, Any]) -> None:
    # """Fail fast, with context, before any network call.

    # Raises:
    #     SignalError: if a required key is missing.
    # """
    missing = [key for key in REQUIRED_KEYS if key not in signal]
    if missing:
        raise SignalError(f"signal missing required keys: {missing}")   

def build_payload(
    signal: Mapping[str, Any],
    *,
    ticker: str = DEFAULT_TICKER,
    test_mode: bool = test_mode,
) -> dict[str, Any]:
    # """Architecture B entry: full position, stop at `stop`, take-profit at the
    # runner target (t2 if a split, else t1). The T1 partial exit and breakeven
    # are fired separately at the T1-fill moment — see partial_exit_t1 / breakeven."""
    validate_signal(signal)

    entry  = signal["entry"]
    action = "buy" if signal["type"] in LONG_SIGNAL_TYPES else "sell"
    # Single-target signals (rr_raw <= 1.1) have t2 = None → target is t1.
    runner_target = signal["t2_points"] if signal.get("t2_points") is not None else signal["t1_points"]

    payload: dict[str, Any] = {
        "test": test_mode,
        "ticker": ticker,
        "action": action,
        "orderType": "limit",
        "price": entry,
        # no-quote brokers (Tradovate/Lucid) require a price on limit/bracket orders
        "signalPrice": entry,
        "quantity": signal["contracts"],
        "stopLoss":   {"type": "stop", "amount": signal["stop_points"]},   
        "takeProfit": {"amount": runner_target},                  
        "extras": {key: signal[key] for key in EXTRA_KEYS if key in signal},
    }
    
    if test_mode:
        payload["test"] = True

    return payload


def _parse_response(resp: requests.Response) -> AlertResult:
    """Map a raw HTTP response onto a structured AlertResult."""
    try:
        body = resp.json()
    except ValueError:
        body = {}

    if resp.status_code == 200:
        return AlertResult(
            ok=True,
            status_code=200,
            signal_id=body.get("id"),
            detail="accepted",
        )

    # 400/404/500: the spec provides messageCode + message on validation errors.
    return AlertResult(
        ok=False,
        status_code=resp.status_code,
        message_code=body.get("messageCode"),
        detail=body.get("message", resp.text),
    )

def traderspost_alert(
    signal: Mapping[str, Any],
    *,
    ticker: str = DEFAULT_TICKER,
    test_mode: bool = test_mode,
    timeout: int = DEFAULT_TIMEOUT,
    webhook_url: str | None = None,
) -> AlertResult:
    # """Send one OTA signal to the TradersPost webhook.

    # Args:
    #     signal:      internal signal mapping.
    #     ticker:      instrument symbol.
    #     test_mode:   if True, TradersPost validates but places no order.
    #     timeout:     seconds before the request is abandoned.
    #     webhook_url: override; defaults to env var TRADERSPOST_WEBHOOK_URL.

    # Returns:
    #     AlertResult — always returned, never raises on network/HTTP failure.
    #     (A SignalError from build_payload is the one intentional exception:
    #     a malformed signal is a programming error, not a runtime condition.)
    # """
    url = webhook_url or os.getenv("TRADERSPOST_WEBHOOK_URL")
    if not url:
        logger.error("TRADERSPOST_WEBHOOK_URL is not set")
        return AlertResult(ok=False, detail="missing webhook url")

    payload = build_payload(signal, ticker=ticker, test_mode=test_mode)
    logger.info("POST payload: %s", json.dumps(payload))

    try:
        resp = requests.post(url, json=payload, timeout=timeout)
    except requests.exceptions.Timeout:
        logger.error("Request timed out after %ss", timeout)
        return AlertResult(ok=False, detail=f"timeout after {timeout}s")
    except requests.exceptions.RequestException as exc:
        logger.error("Request failed: %s", exc)
        return AlertResult(ok=False, detail=str(exc))

    result = _parse_response(resp)
    if result.ok:
        logger.info("Accepted (200) signal_id=%s", result.signal_id)
    else:
        logger.warning(
            "Rejected (%s) code=%s: %s",
            result.status_code,
            result.message_code,
            result.detail,
        )
    return result

    
# ── COMBINED ALERT ────────────────────────────

def send_alert(zone, *, test_mode: bool = test_mode):
    # When ready for real signals to be sent for orders, change test_mode: to bool = True
    # Enforce that alert secrets exist ONLY when actually sending.
    # config.py reads secrets lazily (empty if unset) so backtests/notebooks
    # can import config freely; this is where the requirement is enforced.
    config.require_alert_secrets()
    direction = "BUY  🟢" if zone["type"] == "DBR" else "SELL 🔴"

    subject = f"MYM Signal — {direction} @ {zone['entry']}"

    email_body = f"""
MYM SIGNAL ALERT
─────────────────────────
Direction  : {direction}
Zone Type  : {zone['type']}
Entry      : {zone['entry']}
Stop       : {zone['stop']}
T1 Target  : {zone['t1']}
T2 Target  : {zone['t2']}
Stop Pts   : {zone['stop_points']}
T1 Points  : {zone['t1_points']}
T2 Points  : {zone['t2_points']}
R:R T1     : {zone['rr_t1']}:1
R:R T2     : {zone['rr_t2']}:1
Contracts  : {zone['contracts']}
Score      : {zone['score']}
Formed At  : {zone['formed_at']}
─────────────────────────
Max Risk   : ${zone['total_risk']}
T1 Profit  : ${zone['potential_t1']}
T2 Profit  : ${zone['potential_t2']}
"""
    # Concise push notification
    message = (
        f"{zone['type']} Zone\n"
        f"Entry     : {zone['entry']}\n"
        f"Stop      : {zone['stop']}\n"
        f"T1        : {zone['t1']}\n"
        f"T2        : {zone['t2']}\n"
        f"Formed At : {zone['formed_at']}\n"
        f"Score     : {zone['score']} | R:R {zone['rr_t1']}:1\n"
        f"Risk      : ${zone['total_risk']}\n"
        f"Contracts : {zone['contracts']}\n"  
        f"Stop Pts  : {zone['stop_points']}\n"
        f"T1 Points : {zone['t1_points']}\n"
        f"T2 Points : {zone['t2_points']}\n"
    )
        
    send_email(subject, email_body)
    discord_alert(message, alert_type="signal")
    return traderspost_alert(zone, test_mode=test_mode)
     
# ── TEST FUNCTION ─────────────────────────────

def test_alerts():
    print("Sending test alert...")
    test_zone = {
        "type"        : "DBR",
        "entry"       : 47800,
        "stop"        : 47762,
        "t1"          : 47971,
        "t2"          : 48180,
        "stop_points" : 38,
        "t1_points"   : 171,
        "t2_points"   : 380,
        "rr_t1"       : 4.5,
        "rr_t2"       : 10.0,
        "contracts"   : 2,
        "contracts_t1": 1,
        "contracts_t2": 1,
        "score"       : 9.0,
        "untouched?"  : 0,
        "formed_at"   : "2026-03-24 09:30:00",
        "total_risk"  : 38.00,
        "potential_t1": 171.00,
        "potential_t2": 380.00
    }

    send_alert(test_zone)

if __name__ == "__main__":
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
    )
    test_alerts()

def partial_exit_t1(signal, *, ticker=DEFAULT_TICKER, test_mode=test_mode, timeout=DEFAULT_TIMEOUT):
    """Bank the T1 tranche: partial exit of contracts_t1 on the open position.
    To rest it as a limit at t1 instead of firing at the T1 moment, add
    orderType='limit' + limitPrice=signal['t1'] — VERIFY that combo works first."""
    config.require_alert_secrets()
    url = os.getenv("TRADERSPOST_WEBHOOK_URL")
    if not url:
        return AlertResult(ok=False, detail="missing webhook url")
    payload = {"test": test_mode, "ticker": ticker,
               "action": "exit", "quantity": signal["contracts_t1"]}
    logger.info("T1 PARTIAL EXIT %s", json.dumps(payload))
    try:
        resp = requests.post(url, json=payload, timeout=timeout)
    except requests.exceptions.RequestException as exc:
        return AlertResult(ok=False, detail=str(exc))
    return _parse_response(resp)

def breakeven(ticker=DEFAULT_TICKER, *, test_mode=test_mode, timeout=DEFAULT_TIMEOUT):
    """Move the runner's stop to entry after T1 fills (needs an open position
    and the bracket's stop order)."""
    config.require_alert_secrets()
    url = os.getenv("TRADERSPOST_WEBHOOK_URL")
    if not url:
        return AlertResult(ok=False, detail="missing webhook url")
    payload = {"test": test_mode, "ticker": ticker, "action": "breakeven", "orderType": "stop"}
    logger.info("BREAKEVEN %s", json.dumps(payload))
    try:
        resp = requests.post(url, json=payload, timeout=timeout)
    except requests.exceptions.RequestException as exc:
        return AlertResult(ok=False, detail=str(exc))
    return _parse_response(resp)
    
def flatten(ticker=config.TICKER, *, test_mode=test_mode, timeout=DEFAULT_TIMEOUT):
    """Close any open position for `ticker`. Documented exit: action=exit, no
    quantity (=> full position), ignoreTradingWindows so a window can't block it."""
    config.require_alert_secrets()
    url = os.getenv("TRADERSPOST_WEBHOOK_URL")
    if not url:
        logger.error("TRADERSPOST_WEBHOOK_URL is not set")
        return AlertResult(ok=False, detail="missing webhook url")
    payload = {"ticker": ticker, "action": "exit", "cancel": True, "ignoreTradingWindows": True}
    if test_mode:
        payload["test"] = test_mode
    logger.info("FLATTEN %s", json.dumps(payload))
    try:
        resp = requests.post(url, json=payload, timeout=timeout)
    except requests.exceptions.RequestException as exc:
        logger.error("Flatten failed: %s", exc)
        return AlertResult(ok=False, detail=str(exc))
    return _parse_response(resp)


        # alert_data = {
        #     "type"        : zone["type"],
        #     "entry"       : sets["entry"],
        #     "stop"        : sets["stop"],
        #     "t1"          : sets["t1"],
        #     "t2"          : sets["t2"],
        #     "stop_points" : sets["stop_points"],
        #     "rr_t1"       : sets["rr_t1"],
        #     "rr_t2"       : sets["rr_t2"],
        #     "t1_points"   : sets["t1_points"],
        #     "t2_points"   : sets["t2_points"],
        #     "contracts"   : sets["contracts"],
        #     "score"       : score,
        #     "untouched?"  : zone["touched"],
        #     "formed_at"   : zone["formed_at"],
        #     "total_risk"  : sets["total_risk_$"],
        #     "potential_t1": sets["potential_t1_$"],
        #     "potential_t2": sets["potential_t2_$"],
        # }




In [ ]:
# signals.py — MYM Live Signal Runner (v7-aligned)
# ---------------------------------------------------------------
# Runs ONE detection cycle: load data -> metrics -> context ->
# detect zones -> score -> calculate_sets -> send_alert (email +
# Discord + TradersPost).
#
# STRATEGY FUNCTIONS ARE PORTED 1:1 FROM strategy.py (the v6 build
# that produced the 607-trade / PF 6.08 backtest). Any change to the
# strategy belongs in strategy.py FIRST, re-backtested, then ported
# here — never edited here alone.
#
# Run (from em_algo root):  python -m etrade_stage.signals
# ---------------------------------------------------------------
from __future__ import annotations

import os
import sys
from datetime import datetime

import pandas as pd
import pytz

_base = os.getenv("MYM_BASE", os.path.expanduser("~/prod/em_algo"))
sys.path.insert(0, os.path.join(_base, "task"))

try:                                    # package run: python -m etrade_stage.signals
    from alerts import send_alert
except ImportError:                     # direct run from inside etrade_stage/
    from alerts import send_alert

# ── LOGGING ───────────────────────────────────
def log(msg):
    ts = datetime.now(pytz.timezone(config.TIMEZONE)).strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    os.makedirs(os.path.dirname(config.LOG_FILE), exist_ok=True)
    with open(config.LOG_FILE, "a") as f:
        f.write(line + "\n")

# ── DATA LOADING (active: yfinance) ───────────
def load_from_databento():
    """Live MYM=F bars via yfinance (free, 15-min delay)."""
    from data_feed import load_all_timeframes
    df_5m, df_15m, df_1h = load_all_timeframes()
    log("Live data loaded from Databento (15min delay)")
    return df_5m, df_15m, df_1h

# ── CANDLE METRICS & SESSION (identical to strategy.py) ──
# Create candle metric fields
def add_candle_metrics(df):
    df = df.copy()
    df["range"]     = df["high"] - df["low"]
    df["body"]      = abs(df["close"] - df["open"])
    df["body_pct"]  = (df["body"] / df["range"] * 100).round(1)
    df["direction"] = df.apply(
        lambda row: "BULL" if row["close"] > row["open"] else "BEAR", axis=1
    )
    return df
    
# Create session filter field
def add_session(df):
    df = df.copy()
    df.index = df.index.tz_convert("US/Eastern")
    def get_session(ts):
        hour = ts.hour
        if 7 <= hour < 9: return "NY_PreMarket"
        elif 9 <= hour < 13:  return "NY_AM"
        elif 13 <= hour < 17 : return "NY_PM"    
        elif 3 <= hour < 7: return "LONDON"
        else:               return "ASIA"
    df["session"] = df.index.map(get_session)
    return df

# ── MARKET CONTEXT (v6: curve on 15m/60, trend on 5m/16) ──
def set_curve(df_1h, lookback=config.ZONE_LOOKBACK_DAYS):
    recent        = df_1h.tail(lookback)
    highest_high  = recent["high"].max()
    lowest_low    = recent["low"].min()
    total_range   = highest_high - lowest_low
    upper_third   = highest_high - (total_range * 0.3333)
    lower_third   = lowest_low   + (total_range * 0.3333)
    current_price = df_1h["close"].iloc[-1]
    if current_price >= upper_third:   return "RETAIL"
    elif current_price <= lower_third: return "WHOLESALE"
    else:                              return "NEUTRAL"

def classify_trend(df_15m, window=config.TREND_WINDOW):
    if len(df_15m) < window * 2: return "SIDEWAYS"
    rh   = df_15m["high"].rolling(window).max()
    rl   = df_15m["low"].rolling(window).min()
    ph   = rh.shift(window)
    pl   = rl.shift(window)
    last = df_15m.index[-1]
    if rh[last] > ph[last] and rl[last] > pl[last]:   return "UPTREND"
    elif rh[last] < ph[last] and rl[last] < pl[last]: return "DOWNTREND"
    else:                                             return "SIDEWAYS"

# ── Base Candle & Leg Candle HELPERS FUNCTIONS ───────────────────────────────────────────────────

def is_base_candle(row, avg_range, max_body_pct=35, range_multiplier=0.36):
    if row["range"] == 0: return False
    if row["range"] < 1:  return False
    return (row["body_pct"] <= max_body_pct and
            row["range"]    <= avg_range * range_multiplier)

def is_valid_leg(candle, base_height, min_body_pct=42, min_range_ratio=0.631):
    if base_height == 0: return False
    return (candle["body_pct"] >= min_body_pct and
            candle["range"]    >= base_height * min_range_ratio)
    
# ── ZONE DETECTION (v6 vectorized; floats coerced for JSON) ──
def detect_zones(df, max_base_candles=6, min_body_pct=42,
                 range_multiplier=0.36, min_zone_height=13,
                 min_range_ratio=0.631):
    max_body_pct = 35
    zones = []
    # ── Bug 2 fix: capture datetime index before reset_index() ──
    df          = df.copy()
    df["_dt"]   = df.index
    bars        = df.reset_index(drop=True)
    n           = len(bars)
    ranges      = bars["range"].to_numpy()
    body_pcts   = bars["body_pct"].to_numpy()
    highs       = bars["high"].to_numpy()
    lows        = bars["low"].to_numpy()
    directions  = bars["direction"].to_numpy()
    dts         = bars["_dt"].to_numpy()
    
    # ── Rolling median — robust against log-normal range distribution ──
    avg_rng = df["range"].rolling(50).median().iloc[-1]

    is_base = (
        (ranges != 0) &
        (ranges >= 1) &
        (body_pcts <= max_body_pct) &
        (ranges <= avg_rng * range_multiplier)
    )
    
    # ── Bug 3 fix: start at i=2 — need at least 1 bar before base_start ──
    i = 2
    while i < n - 1:
        if not is_base[i]:
            i += 1; continue

        base_start = base_end = i
        while base_end + 1 < n - 1:
            if is_base[base_end + 1]:
                base_end += 1
            else:
                break

        base_count  = base_end - base_start + 1
        if base_count > max_base_candles:
            i = base_end + 1; continue

        base_high    = highs[base_start:base_end + 1].max()
        base_low    = lows[base_start:base_end + 1].min()
        base_height = base_high - base_low

        if base_height < min_zone_height:
            i = base_end + 1; continue

        leg_in_idx  = base_start - 1
        leg_out_idx = base_end + 1

        if not (body_pcts[leg_in_idx] >= min_body_pct and
                 ranges[leg_in_idx]    >= base_height * min_range_ratio):
             i = base_end + 1; continue
        if not (body_pcts[leg_out_idx] >= min_body_pct and
                 ranges[leg_out_idx]    >= base_height * min_range_ratio):
             i = base_end + 1; continue

        lid = directions[leg_in_idx]
        lod = directions[leg_out_idx]

        if   lid == "BEAR" and lod == "BULL": zone_type = "DBR"
        elif lid == "BULL" and lod == "BEAR": zone_type = "RBD"
        else:
            i = base_end + 1; continue

        # ── Bug 2 fix: use _dt column captured before reset_index() ──
        formed_ts = str(dts[base_end])

        zones.append({
            "type"          : zone_type,
            "proximal"      : round(base_high if zone_type == "DBR" else base_low,  1),
            "distal"        : round(base_low  if zone_type == "DBR" else base_high, 1),
            "base_high"     : round(base_high,        1),
            "base_low"      : round(base_low,         1),
            "base_height"   : round(base_height,      1),
            "base_count"    : base_count,
            "leg_in_range"  : round(ranges[leg_in_idx],  1),
            "leg_out_range" : round(ranges[leg_out_idx], 1),
            "formed_at"     : formed_ts,
            "touched"       : 0,
            "active"        : True
        })

        i = base_end + 2

    return zones

print("detect_zones loaded ✅")

# ── SCORING (identical in strategy.py and here — verified) ──
def score_zone(zone, trend, curve_bias, all_zones):
    score = 0
    # Enhancer 1 — Strength
    move_ratio = zone["leg_out_range"] / zone["base_height"]
    score += 2 if move_ratio >= 2 else 1 if move_ratio >= 1.5 else 0
    
    # Enhancer 2 — Time
    score += 1 if zone["base_count"] <= 3 else 0.5 if zone["base_count"] <= 6 else 0
    
    # Enhancer 3 — Freshness
    score += 2 if zone["touched"] == 0 else 1 if zone["touched"] == 1 else 0
    
    # Enhancer 4 — Trend Alignment
    if (zone["type"] == "DBR" and trend == "UPTREND") or \
       (zone["type"] == "RBD" and trend == "DOWNTREND"):
        score += 1
    else:
        score += 0    # no points   
        
    # Enhancer 5 — Curve Location (Upper/Lower Thirds)
    if (zone["type"] == "DBR" and curve_bias == "WHOLESALE") or \
       (zone["type"] == "RBD" and curve_bias == "RETAIL"):
        score += 2    # aligned with curve — full points
    elif curve_bias == "NEUTRAL":
        score += 1    # neutral zone — partial credit
    else:
        score += 0    # counter-curve — no points (was giving 1 before)
        
    # Enhancer 6 — R:R to nearest opposing zone
    opposing = [z for z in all_zones if z["type"] != zone["type"] and z["active"]]
    if zone["type"] == "DBR":
        above = [z for z in opposing if z["proximal"] > zone["proximal"]]
        if above:
            target = min(above, key=lambda z: z["proximal"])["proximal"]
            risk   = abs(zone["proximal"] - zone["distal"])
            rr     = abs(target - zone["proximal"]) / risk if risk > 0 else 0
            score += 2 if rr >= 5 else 1 if rr >= 3 else 0
    else:
        below = [z for z in opposing if z["proximal"] < zone["proximal"]]
        if below:
            target = max(below, key=lambda z: z["proximal"])["proximal"]
            risk   = abs(zone["proximal"] - zone["distal"])
            rr     = abs(zone["proximal"] - target) / risk if risk > 0 else 0
            score += 2 if rr >= 5 else 1 if rr >= 3 else 0
    return round(score, 1)

print("score_zone loaded ✅")

# ── ORDER CALCULATION (v6 calculate_sets, ported 1:1) ──
def calculate_sets(zone, all_zones, account_size=50000,
                   risk_pct=0.005, tick_value=0.50, stop_buffer=config.STOP_BUFFER,t2_cap=2.5):

    # ── ENTRY & STOP ─────────────────────────────────────────────
    if zone["type"] == "DBR":
        entry = zone["base_high"]
        stop  = zone["base_low"] - stop_buffer
    else:
        entry = zone["base_low"]
        stop  = zone["base_high"] + stop_buffer

    stop_points = abs(entry - stop)
    if stop_points == 0: return None

    direction = 1 if zone["type"] == "DBR" else -1

    # ── POSITION SIZING ──────────────────────────────────────────
    max_risk_dollars  = account_size * risk_pct
    risk_per_contract = stop_points * tick_value
    contracts         = int(max_risk_dollars / risk_per_contract) if risk_per_contract > 0 else 0
    if contracts < 1: return None

    # ── NEAREST OPPOSING ZONE ────────────────────────────────────
    if zone["type"] == "DBR":
        opposing   = [z for z in all_zones
                      if z["type"] == "RBD" and z["proximal"] > entry and z["active"]]
        raw_target = min(opposing, key=lambda z: z["proximal"])["proximal"] if opposing else None
    else:
        opposing   = [z for z in all_zones
                      if z["type"] == "DBR" and z["proximal"] < entry and z["active"]]
        raw_target = max(opposing, key=lambda z: z["proximal"])["proximal"] if opposing else None

    if raw_target is None: return None

    raw_target_pts = abs(raw_target - entry)
    rr_raw         = round(raw_target_pts / stop_points, 2)
    
    # ── TWO-TARGET LOGIC ─────────────────────────────────────────
    if rr_raw < 0.75:
        # Reject — insufficient R:R
        return None
    elif rr_raw <= 1.1:
        # Single target — full position exits at opposing zone
        t1           = raw_target
        t2           = None
        t1_points    = raw_target_pts
        t2_points    = None
        rr_t1        = rr_raw
        rr_t2        = None
        contracts_t1 = contracts
        contracts_t2 = None
    elif rr_raw <= 2.0:
        t1 = round(entry + direction * stop_points * 1.0, 1)
        cap_price = round(entry + direction * stop_points * t2_cap,1)        # 6R cap
        # DOUBLE target — half position exits at t1
        t2 = round(min(raw_target, cap_price), 1) if direction == 1 \
            else round(max(raw_target, cap_price), 1)
        t1_points    = round(abs(t1 - entry), 1)
        t2_points    = round(abs(t2 - entry), 1)
        rr_t1        = round(t1_points / stop_points, 2)
        rr_t2        = round(t2_points / stop_points, 2)
        contracts_t1 = max(1, contracts // 2)
        contracts_t2 = contracts - contracts_t1
    else:
        # Two targets — T1 at 3x stop, T2 capped at opposing zone
        t1 = round(entry + direction * stop_points * 1.5, 1)
        cap_price = round(entry + direction * stop_points * t2_cap,1)        # 6R cap
        #DOUBLE target — half position exits at t2
        t2 = round(min(raw_target, cap_price), 1) if direction == 1 \
            else round(max(raw_target, cap_price), 1)       
        t1_points    = round(abs(t1 - entry), 1)
        t2_points    = round(abs(t2 - entry), 1)
        rr_t1        = round(t1_points / stop_points, 2)
        rr_t2        = round(t2_points / stop_points, 2)
        contracts_t1 = max(1, contracts // 2)
        contracts_t2 = contracts - contracts_t1 if contracts >= 2 else 0
    return {
        "zone_type"           : zone["type"],
        "formed_at"           : zone["formed_at"],
        "entry"               : entry,
        "stop"                : stop,
        "t1"                  : t1,
        "t2"                  : t2,
        "raw_target"          : raw_target,
        "rr_raw"              : rr_raw,
        "stop_points"         : round(stop_points, 1),
        "t1_points"           : t1_points,
        "t2_points"           : t2_points,
        "rr_t1"               : rr_t1,
        "rr_t2"               : rr_t2,
        "contracts"           : contracts,
        "contracts_t1"        : contracts_t1,
        "contracts_t2"        : contracts_t2,
        "risk_per_contract_$" : round(risk_per_contract, 2),
        "total_risk_$"        : round(contracts * risk_per_contract, 2),
        "potential_t1_$"      : round(contracts_t1 * t1_points * tick_value, 2),
        "potential_t2_$"      : round(contracts_t2 * t2_points * tick_value, 2) if contracts_t2 > 0 else 0,
    }

print("calculate_sets loaded ✅")

# ── MAIN SIGNAL CHECK ─────────────────────────
def run_signal_check():
    log("=" * 50)
    log("Running signal check (v7 strategy)...")

    try:
        df_5m, df_15m, df_1h = load_from_databento()
    except Exception as e:
        log(f"Data load failed: {e}")
        return

    df_5m  = add_session(add_candle_metrics(df_5m))
    df_15m = add_session(add_candle_metrics(df_15m))
    df_1h = add_session(add_candle_metrics(df_1h))


    ny = ["NY_AM", "NY_PM", "NY_PreMarket"]
    df_5m_ny  = df_5m[df_5m["session"].isin(ny)].copy()
    df_15m_ny = df_15m[df_15m["session"].isin(ny)].copy()
    df_1h_ny = df_1h[df_1h["session"].isin(ny)].copy()


    if len(df_15m_ny) < 50 or len(df_1h_ny) < config.ZONE_LOOKBACK_DAYS:
        log("Not enough NY-session bars for context — aborting run.")
        return

    # v6 stop buffer: half rolling-median 5m range, capped vs zone height
    stop_buffer = int(min(df_15m_ny["range"].rolling(20).median().iloc[-1] / 2,
                          config.MIN_ZONE_HEIGHT * 0.69))
    log(f"Stop buffer (v7 dynamic, capped): {stop_buffer} pts")

    try:
        curve_bias = set_curve(df_1h_ny)      # v7: curve on 1h
        trend      = classify_trend(df_15m_ny)  # v7: trend on 15m
    except Exception as e:
        log(f"Context failed: {e}")
        return
    log(f"Trend: {trend} | Curve: {curve_bias}")

    # v6: zones detected on the recent window only (matches backtest)
    zone_cutoff = df_15m_ny.index.max() - pd.Timedelta(days=config.ZONE_WINDOW_DAYS)
    zones = detect_zones(df_15m_ny[df_15m_ny.index >= zone_cutoff])
    log(f"Zones detected (last {config.ZONE_WINDOW_DAYS}d): {len(zones)}")

    signals_sent = 0
    for zone in zones:
        score = score_zone(zone, trend, curve_bias, zones)
        if score < config.MIN_SCORE:
            continue

        sets = calculate_sets(zone, zones, stop_buffer=stop_buffer)
        if sets is None:
            continue

        alert_data = {
            "type"        : zone["type"],
            "entry"       : sets["entry"],
            "stop"        : sets["stop"],
            "t1"          : sets["t1"],
            "t2"          : sets["t2"],
            "stop_points" : sets["stop_points"],
            "rr_t1"       : sets["rr_t1"],
            "rr_t2"       : sets["rr_t2"],
            "t1_points"   : sets["t1_points"],
            "t2_points"   : sets["t2_points"],
            "contracts"   : sets["contracts"],
            "score"       : score,
            "untouched?"  : zone["touched"],
            "formed_at"   : zone["formed_at"],
            "total_risk"  : sets["total_risk_$"],
            "potential_t1": sets["potential_t1_$"],
            "potential_t2": sets["potential_t2_$"],
        }

        log(f"Signal: {zone['type']} @ {sets['entry']} | Score {score} | "
            f"R:R T1 {sets['rr_t1']} | {sets['contracts']} contracts | "
            f"risk ${sets['total_risk_$']}")
        result = send_alert(alert_data)        # test_mode default=True in alerts.py
        log(f"TradersPost: ok={result.ok} id={result.signal_id} detail={result.detail}")
        signals_sent += 1

    if signals_sent == 0:
        log("No qualifying signals found this run.")
    log(f"Done. {signals_sent} signal(s) sent.")
    log("=" * 50)

if __name__ == "__main__":
    run_signal_check()
